# Teapot DSPL Workflow

这个 notebook 参考 `Herculens/Slicelens/Herculens_3.ipynb` 的顺序来组织：先读图和可视化，再定义统一的 `parametric/pixelated` 模型接口，最后各跑一个 `1-step` smoke test。

注意：旧文件删除还没有执行，因为按当前工作规则需要额外确认。

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'Tian_infra.py').exists():
    for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
        if (candidate / 'Tian_infra.py').exists():
            NOTEBOOK_DIR = candidate
            break
if not (NOTEBOOK_DIR / 'Tian_infra.py').exists():
    raise FileNotFoundError('Cannot locate Tian_infra.py from the current working directory.')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from Tian_infra import import_function_DSPL
import_function_DSPL(globals())

NOTEBOOK_DIR


In [ ]:
PREPROCESSED_DIR = NOTEBOOK_DIR / 'preprocessed' / 'prep_20260419_bgbox_psfsvi_v2_flat'
PREPROCESS_META_PATH = PREPROCESSED_DIR / 'preprocessing_metadata.json'
PREPROCESS_META = json.loads(PREPROCESS_META_PATH.read_text())
CONJUGATE_JSON_PATH = PREPROCESSED_DIR / 'f814w_lens_cutout_100x100_conjugate_points.json'
CONJUGATE_PAYLOAD = json.loads(CONJUGATE_JSON_PATH.read_text())

PIXEL_GRID_SHAPE = 12
PARAMETRIC_STEPS = 1
PIXELATED_STEPS = 1
STAGE2_INIT_STEPS = 200
RNG_SEED = 7
N_GAUSS_LENS = 5
N_GAUSS_SOURCE = 1
THETA_E_LOW = 0.2
THETA_E_HIGH = 2.0
SIS_THETA_LOW = 0.02
SIS_THETA_HIGH = 0.6
Q_PRIOR = (0.4, 0.98)
PHI_PRIOR = (-0.5 * np.pi, 0.5 * np.pi)
CENTER_PRIOR = (-0.4, 0.4)
SOURCE_GRID_SCALE = 1.0
ETA_PRIOR = (1.0, 2.0)
REFERENCE_BAND = 'f814w'
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'teapot_notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CUTOUT_SIZE = tuple(PREPROCESS_META['science_cutout_size'])
INNER_MASK_PATH = PREPROCESSED_DIR / 'f814w_science_100x100_mask_inner.fits'
OUTER_MASK_PATH = PREPROCESSED_DIR / 'f475w_science_100x100_mask outer.fits'

BAND_FILES = {
    'f475w': {
        'science_path': PREPROCESSED_DIR / 'f475w_lens_cutout_100x100.fits',
        'psf_path': PREPROCESSED_DIR / 'f475w_psf_model_svi.fits',
        'background_box_path': PREPROCESSED_DIR / 'f475w_background_box_50x50.fits',
    },
    'f814w': {
        'science_path': PREPROCESSED_DIR / 'f814w_lens_cutout_100x100.fits',
        'psf_path': PREPROCESSED_DIR / 'f814w_psf_model_svi.fits',
        'background_box_path': PREPROCESSED_DIR / 'f814w_background_box_50x50.fits',
    },
}

with fits.open(BAND_FILES['f475w']['psf_path'], memmap=True) as hdul:
    PSF_CUTOUT_SIZE = tuple(np.asarray(hdul[0].data).shape)

for band_key, band_files in BAND_FILES.items():
    for file_key, file_path in band_files.items():
        if not file_path.exists():
            raise FileNotFoundError(f'Missing {band_key} {file_key}: {file_path}')

for band_key, band_meta in PREPROCESS_META['cutouts'].items():
    print(f"{band_key} science center pix: {tuple(band_meta['center_pix'])}")
    print(f"{band_key} science path: {BAND_FILES[band_key]['science_path']}")
    print(f"{band_key} psf path: {BAND_FILES[band_key]['psf_path']}")
print('eta prior:', ETA_PRIOR)
print('inner mask:', INNER_MASK_PATH)
print('outer mask:', OUTER_MASK_PATH)
print('conjugate json:', CONJUGATE_JSON_PATH)


In [ ]:
@dataclass
class BandData:
    key: str
    filter_name: str
    data: jnp.ndarray
    obs_mask: jnp.ndarray
    num_obs: int
    source1_mask: np.ndarray
    source2_mask: np.ndarray
    rms: float
    pixel_scale: float
    exposure_time: float
    science_cutout_path: Path
    source1_mask_cutout_path: Path
    source2_mask_cutout_path: Path
    psf_cutout_path: Path
    background_box_path: Path
    lens_image_parametric: MPLensImage
    lens_image_pixelated: MPLensImage


def pixel_scale_arcsec(header):
    if 'PIXSCALE' in header:
        return float(header['PIXSCALE'])
    if 'D001SCAL' in header:
        return float(header['D001SCAL'])
    cd11 = header.get('CD1_1')
    cd21 = header.get('CD2_1')
    if cd11 is not None and cd21 is not None:
        return float(np.hypot(cd11, cd21) * 3600.0)
    raise ValueError('Cannot determine pixel scale from FITS header.')


def load_binary_mask(path):
    with fits.open(path, memmap=True) as hdul:
        mask = np.asarray(hdul[0].data, dtype=np.uint8)
    return mask > 0


SOURCE1_MASK = load_binary_mask(INNER_MASK_PATH)
SOURCE2_MASK = load_binary_mask(OUTER_MASK_PATH)
if SOURCE1_MASK.shape != tuple(CUTOUT_SIZE):
    raise ValueError(f'Inner mask shape {SOURCE1_MASK.shape} does not match cutout {CUTOUT_SIZE}')
if SOURCE2_MASK.shape != tuple(CUTOUT_SIZE):
    raise ValueError(f'Outer mask shape {SOURCE2_MASK.shape} does not match cutout {CUTOUT_SIZE}')


def build_band_data(key):
    band_meta = PREPROCESS_META['cutouts'][key]
    band_files = BAND_FILES[key]
    science_cutout_path = band_files['science_path']
    psf_cutout_path = band_files['psf_path']
    background_box_path = band_files['background_box_path']

    with fits.open(science_cutout_path, memmap=True) as hdul:
        science_header = hdul[0].header
        science_data = np.asarray(hdul[0].data, dtype=np.float64)
    with fits.open(psf_cutout_path, memmap=True) as hdul:
        psf_header = hdul[0].header
        psf_kernel = np.asarray(hdul[0].data, dtype=np.float64)

    filter_name = science_header.get('FILTER', key.upper())
    exposure_time = float(band_meta['exptime_sec'])
    pixel_scale = pixel_scale_arcsec(science_header)
    rms = float(band_meta['background_sigma_scalar'])

    obs_mask = np.isfinite(science_data)
    science_data = np.nan_to_num(science_data, nan=0.0).astype(np.float64)
    source1_mask = np.asarray(SOURCE1_MASK, dtype=bool)
    source2_mask = np.asarray(SOURCE2_MASK, dtype=bool)

    psf_kernel = np.nan_to_num(psf_kernel, nan=0.0).astype(np.float64)
    psf_kernel = np.clip(psf_kernel, a_min=0.0, a_max=None)
    psf_kernel /= psf_kernel.sum()

    pixel_grid = Geometry.get_pixel_grid(science_data, pixel_scale)[0]
    psf = PSF(psf_type='PIXEL', kernel_point_source=psf_kernel)
    noise = Noise(
        pixel_grid.num_pixel_axes[1],
        pixel_grid.num_pixel_axes[0],
        exposure_time=exposure_time,
        background_rms=rms,
    )
    source_arc_masks = [None, source1_mask, source2_mask]
    source_scales = [1.0, SOURCE_GRID_SCALE, SOURCE_GRID_SCALE]

    parametric_light_model = MPLightModel([
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
    ])
    pixel_kwargs = {
        'pixel_adaptive_grid': True,
        'pixel_interpol': 'fast_bilinear',
        'kwargs_pixelated': {'num_pixels': PIXEL_GRID_SHAPE},
    }
    pixelated_light_model = MPLightModel([
        LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
        LightModel(['PIXELATED'], **pixel_kwargs),
        LightModel(['PIXELATED'], **pixel_kwargs),
    ])
    mass_model = MPMassModel([
        MassModel(['EPL', 'SHEAR']),
        MassModel(['SIS']),
    ])
    lens_image_parametric = MPLensImage(
        pixel_grid,
        psf,
        noise_class=noise,
        light_model_class=parametric_light_model,
        mass_model_class=mass_model,
        source_arc_masks=source_arc_masks,
        source_grid_scale=source_scales,
        kwargs_numerics={'supersampling_factor': 1},
    )
    lens_image_pixelated = MPLensImage(
        pixel_grid,
        psf,
        noise_class=noise,
        light_model_class=pixelated_light_model,
        mass_model_class=mass_model,
        source_arc_masks=source_arc_masks,
        source_grid_scale=source_scales,
        kwargs_numerics={'supersampling_factor': 1},
    )
    return BandData(
        key=key,
        filter_name=filter_name,
        data=jnp.asarray(science_data, dtype=jnp.float64),
        obs_mask=jnp.asarray(obs_mask, dtype=bool),
        num_obs=int(obs_mask.sum()),
        source1_mask=source1_mask,
        source2_mask=source2_mask,
        rms=rms,
        pixel_scale=pixel_scale,
        exposure_time=exposure_time,
        science_cutout_path=science_cutout_path,
        source1_mask_cutout_path=INNER_MASK_PATH,
        source2_mask_cutout_path=OUTER_MASK_PATH,
        psf_cutout_path=psf_cutout_path,
        background_box_path=background_box_path,
        lens_image_parametric=lens_image_parametric,
        lens_image_pixelated=lens_image_pixelated,
    )


In [ ]:
K_GRID_PIXEL = PowerSpectrum.K_grid((PIXEL_GRID_SHAPE, PIXEL_GRID_SHAPE))


def scalar_param(params, key):
    return jnp.ravel(jnp.asarray(params[key], dtype=jnp.float64))[0]


def vector_param(params, key):
    return jnp.asarray(params[key], dtype=jnp.float64)


def rotation_matrix(phi):
    c = jnp.cos(phi)
    s = jnp.sin(phi)
    return jnp.array([[c, -s], [s, c]], dtype=jnp.float64)


def qphi_to_e1e2(q, phi):
    ellipticity = (1.0 - q) / (1.0 + q)
    return ellipticity * jnp.cos(2.0 * phi), ellipticity * jnp.sin(2.0 * phi)


def get_band(band_key):
    for band in bands:
        if band.key == band_key:
            return band
    raise KeyError(f'Unknown band key: {band_key}')


def get_conjugate_centroid_display(payload):
    points = payload.get('conjugatePointsDisplayCoordinates', [])
    if len(points) == 0:
        raise ValueError(f'No conjugate points found in {CONJUGATE_JSON_PATH}')
    xy = np.array([[point['x'], point['y']] for point in points], dtype=float)
    return jnp.asarray(xy.mean(axis=0), dtype=jnp.float64)


def pixel_display_to_model_xy(point_xy, band):
    point_xy = jnp.asarray(point_xy, dtype=jnp.float64)
    nx = band.data.shape[1]
    ny = band.data.shape[0]
    center_xy = jnp.array([(nx - 1) / 2.0, (ny - 1) / 2.0], dtype=jnp.float64)
    return (point_xy - center_xy) * band.pixel_scale


def get_reference_conjugate_point_model(reference_band_key=REFERENCE_BAND):
    reference_band = get_band(reference_band_key)
    centroid_display = get_conjugate_centroid_display(CONJUGATE_PAYLOAD)
    return pixel_display_to_model_xy(centroid_display, reference_band)


def sample_shared_main_mass_shape():
    return {
        'theta_E_main': numpyro.sample('theta_E_main', dist.Uniform(THETA_E_LOW, THETA_E_HIGH)),
        'gamma_main': numpyro.sample('gamma_main', dist.Uniform(1.6, 2.4)),
        'q_main': numpyro.sample('q_main', dist.Uniform(*Q_PRIOR)),
        'gamma_shear_main': numpyro.sample('gamma_shear_main', dist.Uniform(-0.15, 0.15).expand([2])),
    }


def sample_parametric_main_mass_params():
    params = sample_shared_main_mass_shape()
    params['center_main_shared'] = numpyro.sample(
        'center_main_shared',
        dist.TruncatedNormal(0.0, 0.08, low=CENTER_PRIOR[0], high=CENTER_PRIOR[1]).expand([2]),
    )
    params['phi_main_shared'] = numpyro.sample('phi_main_shared', dist.Uniform(*PHI_PRIOR))
    return params


def sample_pixelated_main_mass_params():
    params = sample_shared_main_mass_shape()
    for band in ['f475w', 'f814w']:
        params[f'center_main_{band}'] = numpyro.sample(
            f'center_main_{band}',
            dist.TruncatedNormal(0.0, 0.12, low=CENTER_PRIOR[0], high=CENTER_PRIOR[1]).expand([2]),
        )
        params[f'phi_main_{band}'] = numpyro.sample(f'phi_main_{band}', dist.Uniform(*PHI_PRIOR))
    return params


def get_main_center_phi(params, band_key, model_mode):
    if model_mode == 'parametric':
        return vector_param(params, 'center_main_shared'), scalar_param(params, 'phi_main_shared')
    return vector_param(params, f'center_main_{band_key}'), scalar_param(params, f'phi_main_{band_key}')


def build_main_mass_kwargs(params, band_key, model_mode):
    center_xy, phi = get_main_center_phi(params, band_key, model_mode)
    e1, e2 = qphi_to_e1e2(scalar_param(params, 'q_main'), phi)
    shear = vector_param(params, 'gamma_shear_main')
    return [{
        'theta_E': scalar_param(params, 'theta_E_main'),
        'gamma': scalar_param(params, 'gamma_main'),
        'e1': e1,
        'e2': e2,
        'center_x': center_xy[0],
        'center_y': center_xy[1],
    }, {
        'gamma1': shear[0],
        'gamma2': shear[1],
        'ra_0': center_xy[0],
        'dec_0': center_xy[1],
    }]


def transform_reference_point_to_band(point_reference, center_reference, phi_reference, center_target, phi_target):
    intrinsic_offset = rotation_matrix(-phi_reference) @ (point_reference - center_reference)
    return center_target + rotation_matrix(phi_target) @ intrinsic_offset


def build_source1_sis_kwargs(params, band_key, model_mode):
    point_reference = get_reference_conjugate_point_model(REFERENCE_BAND)
    center_reference, phi_reference = get_main_center_phi(params, REFERENCE_BAND, model_mode)
    center_target, phi_target = get_main_center_phi(params, band_key, model_mode)
    sis_center = transform_reference_point_to_band(
        point_reference,
        center_reference,
        phi_reference,
        center_target,
        phi_target,
    )
    return [{
        'theta_E': scalar_param(params, 'theta_E_source1'),
        'center_x': sis_center[0],
        'center_y': sis_center[1],
    }]


def build_mass_kwargs(params, band_key, model_mode):
    return [
        build_main_mass_kwargs(params, band_key, model_mode),
        build_source1_sis_kwargs(params, band_key, model_mode),
    ]


def build_eta_flat(params):
    return jnp.atleast_1d(jnp.asarray(params['eta'], dtype=jnp.float64))


def pixelize_parametric_source(band, params, plane_index):
    kwargs_mass = build_mass_kwargs(params, band.key, 'parametric')
    kwargs_light = Light.band_from_parametric_params(params, band.key)
    eta_flat = build_eta_flat(params)
    x_axes, y_axes, _ = band.lens_image_pixelated.get_source_coordinates(
        eta_flat,
        kwargs_mass,
        force=True,
        npix_src=PIXEL_GRID_SHAPE,
        source_grid_scale=SOURCE_GRID_SCALE,
    )
    x_grid, y_grid = jnp.meshgrid(x_axes[plane_index], y_axes[plane_index])
    source_image = band.lens_image_parametric.MPLightModel.light_models[plane_index].surface_brightness(
        x_grid,
        y_grid,
        kwargs_light[plane_index],
        pixels_x_coord=x_axes[plane_index],
        pixels_y_coord=y_axes[plane_index],
    )
    return source_image * band.lens_image_parametric.Grid.pixel_area


def fit_power_spectrum_init(image, rng_key, param_name, max_iterations=STAGE2_INIT_STEPS):
    image = jnp.asarray(image, dtype=jnp.float64)
    noise = jnp.maximum(1e-6, 0.001 * jnp.max(image))

    def power_spectrum_init_model(image_in, noise_in, k_values):
        ps_model = PowerSpectrum.matern_power_spectrum(
            f'Init {param_name}',
            param_name,
            k_values,
            n_value=None,
            positive=True,
        )
        ny, nx = image_in.shape
        with numpyro.plate(f'{param_name} data x - [{nx}]', nx):
            with numpyro.plate(f'{param_name} data y - [{ny}]', ny):
                numpyro.sample(
                    f'obs_{param_name}',
                    dist.Normal(ps_model['pixels'], noise_in),
                    obs=image_in,
                )

    guide = autoguide.AutoDiagonalNormal(power_spectrum_init_model, init_loc_fn=infer.init_to_median())
    scheduler = split_scheduler(
        max_iterations,
        init_value=0.01,
        transition_steps=[50, 10],
    )
    svi = infer.SVI(
        power_spectrum_init_model,
        guide,
        optax.adabelief(learning_rate=scheduler),
        infer.TraceMeanField_ELBO(),
    )
    result = svi.run(
        rng_key,
        max_iterations,
        image,
        noise,
        K_GRID_PIXEL.k,
        progress_bar=False,
        stable_update=True,
    )
    return guide.median(result.params)


def build_stage2_init_values(parametric_params, rng_key):
    stage2_init_values = {
        'eta': parametric_params['eta'],
        'theta_E_main': parametric_params['theta_E_main'],
        'gamma_main': parametric_params['gamma_main'],
        'q_main': parametric_params['q_main'],
        'gamma_shear_main': parametric_params['gamma_shear_main'],
        'theta_E_source1': parametric_params['theta_E_source1'],
        'center_main_f475w': parametric_params['center_main_shared'],
        'center_main_f814w': parametric_params['center_main_shared'],
        'phi_main_f475w': parametric_params['phi_main_shared'],
        'phi_main_f814w': parametric_params['phi_main_shared'],
    }
    for band in bands:
        stage2_init_values[f'RMS_{band.key}'] = parametric_params[f'RMS_{band.key}']
    fit_key = rng_key
    for band in bands:
        for plane_index, param_name in [
            (1, f'source1pix_{band.key}'),
            (2, f'source2pix_{band.key}'),
        ]:
            fit_key, ps_key = jax.random.split(fit_key)
            source_image = pixelize_parametric_source(band, parametric_params, plane_index)
            ps_init = fit_power_spectrum_init(source_image, ps_key, param_name)
            stage2_init_values.update(ps_init)
    return stage2_init_values


def build_parametric_model():
    sigma_lims_lens = (0.01, 1.0)
    sigma_lims_source = (0.01, 0.5)

    def model():
        model_params = sample_parametric_main_mass_params()
        model_params['eta'] = numpyro.sample('eta', dist.Uniform(*ETA_PRIOR))
        model_params['theta_E_source1'] = numpyro.sample('theta_E_source1', dist.Uniform(SIS_THETA_LOW, SIS_THETA_HIGH))
        eta_flat = build_eta_flat(model_params)

        for band in bands:
            kwargs_mass = build_mass_kwargs(model_params, band.key, 'parametric')
            lens_light = Light.multi_gauss_light(
                f'{band.key} lens light',
                f'lens_{band.key}',
                N_GAUSS_LENS,
                sigma_lims_lens,
                center_low=-0.2,
                center_high=0.2,
            )
            source_1_light = Light.multi_gauss_light(
                f'{band.key} source 1 light',
                f'source1_{band.key}',
                N_GAUSS_SOURCE,
                sigma_lims_source,
                center_low=-0.3,
                center_high=0.3,
            )
            source_2_light = Light.multi_gauss_light(
                f'{band.key} source 2 light',
                f'source2_{band.key}',
                N_GAUSS_SOURCE,
                sigma_lims_source,
                center_low=-0.4,
                center_high=0.4,
            )
            kwargs_light = [lens_light, source_1_light, source_2_light]
            model_image = band.lens_image_parametric.model(
                eta_flat=eta_flat,
                kwargs_mass=kwargs_mass,
                kwargs_light=kwargs_light,
            )
            rms_low = max(0.5 * band.rms, 1e-6)
            rms_high = max(2.0 * band.rms, rms_low * 1.5)
            background_rms = numpyro.sample(f'RMS_{band.key}', dist.LogUniform(rms_low, rms_high))
            model_var = band.lens_image_parametric.Noise.C_D_model(model_image, background_rms=background_rms)
            model_std = jnp.sqrt(jnp.maximum(model_var, 1e-12))
            numpyro.deterministic(f'model_{band.key}', model_image)
            with numpyro.plate(f'obs_{band.key} - [{band.num_obs}]', band.num_obs):
                numpyro.sample(
                    f'obs_{band.key}',
                    dist.Normal(model_image[band.obs_mask], model_std[band.obs_mask]),
                    obs=band.data[band.obs_mask],
                )

    return model


def build_pixelated_model(fixed_lens_lights):
    def model():
        model_params = sample_pixelated_main_mass_params()
        model_params['eta'] = numpyro.sample('eta', dist.Uniform(*ETA_PRIOR))
        model_params['theta_E_source1'] = numpyro.sample('theta_E_source1', dist.Uniform(SIS_THETA_LOW, SIS_THETA_HIGH))
        eta_flat = build_eta_flat(model_params)

        for band in bands:
            kwargs_mass = build_mass_kwargs(model_params, band.key, 'pixelated')
            source_1_light = PowerSpectrum.matern_power_spectrum(
                f'{band.key} source 1 power',
                f'source1pix_{band.key}',
                K_GRID_PIXEL.k,
                k_zero=0.0,
            )
            source_2_light = PowerSpectrum.matern_power_spectrum(
                f'{band.key} source 2 power',
                f'source2pix_{band.key}',
                K_GRID_PIXEL.k,
                k_zero=0.0,
            )
            kwargs_light = [
                fixed_lens_lights[band.key],
                [source_1_light],
                [source_2_light],
            ]
            model_image = band.lens_image_pixelated.model(
                eta_flat=eta_flat,
                kwargs_mass=kwargs_mass,
                kwargs_light=kwargs_light,
            )
            rms_low = max(0.5 * band.rms, 1e-6)
            rms_high = max(2.0 * band.rms, rms_low * 1.5)
            background_rms = numpyro.sample(f'RMS_{band.key}', dist.LogUniform(rms_low, rms_high))
            model_var = band.lens_image_pixelated.Noise.C_D_model(model_image, background_rms=background_rms)
            model_std = jnp.sqrt(jnp.maximum(model_var, 1e-12))
            numpyro.deterministic(f'model_{band.key}', model_image)
            with numpyro.plate(f'obs_{band.key} - [{band.num_obs}]', band.num_obs):
                numpyro.sample(
                    f'obs_{band.key}',
                    dist.Normal(model_image[band.obs_mask], model_std[band.obs_mask]),
                    obs=band.data[band.obs_mask],
                )

    return model


def build_model(model_mode, bands, fixed_lens_lights=None):
    if model_mode == 'parametric':
        return build_parametric_model()
    if model_mode == 'pixelated':
        if fixed_lens_lights is None:
            raise ValueError('fixed_lens_lights is required for pixelated mode')
        return build_pixelated_model(fixed_lens_lights)
    raise ValueError(f'Unknown model_mode: {model_mode}')


def run_svi(model, rng_key, steps, init_values=None):
    init_fn = infer.init_to_median(num_samples=5)
    if init_values:
        init_fn = ResumeInit.init_to_value_or_defer(values=init_values, defer=init_fn)
    guide = autoguide.AutoLowRankMultivariateNormal(model, init_loc_fn=init_fn)
    scheduler = split_scheduler(
        max_iterations=max(steps, 2),
        init_value=0.01,
        decay_rates=[0.95, 0.98],
        transition_steps=[20, 5],
        boundary=0.8,
    )
    optim = optax.adam(learning_rate=scheduler)
    svi = infer.SVI(model, guide, optim, infer.Trace_ELBO())
    result = svi.run(rng_key, steps, progress_bar=False, stable_update=True)
    return result, guide.median(result.params)


def build_light_kwargs_from_params(params, band_key, model_mode, fixed_lens_lights=None):
    if model_mode == 'parametric':
        return Light.band_from_parametric_params(params, band_key)
    return Light.band_from_pixelated_params(params, band_key, fixed_lens_lights)


def plot_stage_results(title, model_mode, params, bands, fixed_lens_lights=None):
    fig, axes = plt.subplots(len(bands), 3, figsize=(12, 4 * len(bands)))
    if len(bands) == 1:
        axes = np.array([axes])
    for row, band in enumerate(bands):
        lens_image = band.lens_image_parametric if model_mode == 'parametric' else band.lens_image_pixelated
        kwargs_mass = build_mass_kwargs(params, band.key, model_mode)
        kwargs_light = build_light_kwargs_from_params(params, band.key, model_mode, fixed_lens_lights=fixed_lens_lights)
        model_image = np.asarray(lens_image.model(
            eta_flat=build_eta_flat(params),
            kwargs_mass=kwargs_mass,
            kwargs_light=kwargs_light,
        ))
        residual = np.asarray(band.data) - model_image

        for col, image, cmap, norm in [
            (0, np.asarray(band.data), 'twilight', LogNorm(vmin=max(np.nanpercentile(np.asarray(band.data)[np.asarray(band.data) > 0], 5), 1e-6), vmax=max(np.nanpercentile(np.asarray(band.data)[np.asarray(band.data) > 0], 99.5), 1e-5)) if np.any(np.asarray(band.data) > 0) else None),
            (1, model_image, 'twilight', LogNorm(vmin=max(np.nanpercentile(model_image[model_image > 0], 5), 1e-6), vmax=max(np.nanpercentile(model_image[model_image > 0], 99.5), 1e-5)) if np.any(model_image > 0) else None),
            (2, residual, 'coolwarm', None),
        ]:
            axes[row, col].imshow(image, origin='lower', cmap=cmap, norm=norm)
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
        axes[row, 0].set_title(f'{band.filter_name} data')
        axes[row, 1].set_title(f'{band.filter_name} model')
        axes[row, 2].set_title(f'{band.filter_name} residual')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def main_mass_summary(params, model_mode):
    summary = {
        'theta_E': float(np.asarray(scalar_param(params, 'theta_E_main'))),
        'gamma': float(np.asarray(scalar_param(params, 'gamma_main'))),
        'q': float(np.asarray(scalar_param(params, 'q_main'))),
        'gamma1': float(np.asarray(vector_param(params, 'gamma_shear_main')[0])),
        'gamma2': float(np.asarray(vector_param(params, 'gamma_shear_main')[1])),
        'bands': {},
    }
    for band in ['f475w', 'f814w']:
        center_xy, phi = get_main_center_phi(params, band, model_mode)
        sis_kwargs = build_source1_sis_kwargs(params, band, model_mode)[0]
        summary['bands'][band] = {
            'center_x': float(np.asarray(center_xy[0])),
            'center_y': float(np.asarray(center_xy[1])),
            'phi_rad': float(np.asarray(phi)),
            'phi_deg': float(np.asarray(phi) * 180.0 / np.pi),
            'source1_sis_center_x': float(np.asarray(sis_kwargs['center_x'])),
            'source1_sis_center_y': float(np.asarray(sis_kwargs['center_y'])),
        }
    return summary


In [ ]:
bands = [
    build_band_data('f475w'),
    build_band_data('f814w'),
]
conjugate_centroid_display = np.asarray(get_conjugate_centroid_display(CONJUGATE_PAYLOAD), dtype=float)
conjugate_centroid_model_f814 = np.asarray(get_reference_conjugate_point_model(), dtype=float)

fig, axes = plt.subplots(len(bands), 2, figsize=(10, 4 * len(bands)))
if len(bands) == 1:
    axes = np.array([axes])
for row, band in enumerate(bands):
    science_image = np.asarray(band.data, dtype=float)
    finite_science = science_image[np.isfinite(science_image) & (science_image > 0)]
    science_vmin = max(float(np.nanpercentile(finite_science, 5)), 1e-6) if finite_science.size else 1e-6
    science_vmax = max(float(np.nanpercentile(finite_science, 99.5)), science_vmin * 1.01) if finite_science.size else 1.0
    axes[row, 0].imshow(
        science_image,
        origin='lower',
        cmap='twilight',
        norm=LogNorm(vmin=science_vmin, vmax=science_vmax),
    )
    axes[row, 0].contour(band.source1_mask.astype(float), levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--')
    axes[row, 0].contour(band.source2_mask.astype(float), levels=[0.5], colors='orange', linewidths=1.0, linestyles='--')
    if band.key == REFERENCE_BAND:
        axes[row, 0].scatter(conjugate_centroid_display[0], conjugate_centroid_display[1], s=20, c='yellow', marker='x')
    axes[row, 0].set_title(f'{band.filter_name} science')
    with fits.open(band.psf_cutout_path) as hdul:
        psf_image = np.asarray(hdul[0].data, dtype=float)
        finite_psf = psf_image[np.isfinite(psf_image) & (psf_image > 0)]
        psf_vmin = max(float(np.nanpercentile(finite_psf, 5)), 1e-8) if finite_psf.size else 1e-8
        psf_vmax = max(float(np.nanpercentile(finite_psf, 99.5)), psf_vmin * 1.01) if finite_psf.size else 1.0
        axes[row, 1].imshow(
            psf_image,
            origin='lower',
            cmap='twilight',
            norm=LogNorm(vmin=psf_vmin, vmax=psf_vmax),
        )
    axes[row, 1].set_title(f'{band.filter_name} PSF')
plt.tight_layout()
plt.show()
for band in bands:
    print(band.filter_name)
    print('  science   :', band.science_cutout_path)
    print('  source1   :', band.source1_mask_cutout_path)
    print('  source2   :', band.source2_mask_cutout_path)
    print('  psf       :', band.psf_cutout_path)
    print('  background:', band.background_box_path)
print('reference conjugate centroid display (F814W pix):', conjugate_centroid_display.tolist())
print('reference conjugate centroid model (F814W arcsec):', conjugate_centroid_model_f814.tolist())


In [ ]:
MODEL_MODE = 'parametric'
rng_key = jax.random.PRNGKey(RNG_SEED)
rng_key, stage1_key, stage2_init_key, stage2_key = jax.random.split(rng_key, 4)
model_parametric = build_model(MODEL_MODE, bands)
parametric_result, parametric_median = run_svi(model_parametric, stage1_key, PARAMETRIC_STEPS)
fixed_lens_lights = {
    band.key: Light.band_from_parametric_params(parametric_median, band.key)[0]
    for band in bands
}
print('parametric loss =', float(parametric_result.losses[-1]))
print('parametric eta  =', float(np.asarray(parametric_median['eta'])))


In [ ]:
plot_stage_results(
    title='Parametric DSPL Smoke Test',
    model_mode='parametric',
    params=parametric_median,
    bands=bands,
)


In [ ]:
MODEL_MODE = 'pixelated'
model_pixelated = build_model(MODEL_MODE, bands, fixed_lens_lights=fixed_lens_lights)
stage2_init_values = build_stage2_init_values(parametric_median, stage2_init_key)
pixelated_result, pixelated_median = run_svi(model_pixelated, stage2_key, PIXELATED_STEPS, init_values=stage2_init_values)
print('pixelated loss =', float(pixelated_result.losses[-1]))
print('pixelated eta  =', float(np.asarray(pixelated_median['eta'])))


In [ ]:
plot_stage_results(
    title='Pixelated DSPL Smoke Test',
    model_mode='pixelated',
    params=pixelated_median,
    bands=bands,
    fixed_lens_lights=fixed_lens_lights,
)
summary = {
    'cutout_size': CUTOUT_SIZE,
    'psf_cutout_size': list(PSF_CUTOUT_SIZE),
    'pixel_grid_shape': PIXEL_GRID_SHAPE,
    'eta_prior': {'low': ETA_PRIOR[0], 'high': ETA_PRIOR[1]},
    'eta_parametric': float(np.asarray(parametric_median['eta'])),
    'eta_pixelated': float(np.asarray(pixelated_median['eta'])),
    'parametric_loss_final': float(parametric_result.losses[-1]),
    'pixelated_loss_final': float(pixelated_result.losses[-1]),
    'reference_band': REFERENCE_BAND,
    'conjugate_json_path': str(CONJUGATE_JSON_PATH),
    'conjugate_centroid_display_xy': list(map(float, np.asarray(get_conjugate_centroid_display(CONJUGATE_PAYLOAD)))),
    'conjugate_centroid_model_f814_arcsec': list(map(float, np.asarray(get_reference_conjugate_point_model()))),
    'bands': {
        band.key: {
            'filter': band.filter_name,
            'rms': band.rms,
            'pixel_scale': band.pixel_scale,
            'science_center_pix': list(PREPROCESS_META['cutouts'][band.key]['center_pix']),
            'background_box_center_pix': [
                PREPROCESS_META['cutouts'][band.key]['background_box_info']['center_x'],
                PREPROCESS_META['cutouts'][band.key]['background_box_info']['center_y'],
            ],
            'science_cutout_path': str(band.science_cutout_path),
            'source1_mask_cutout_path': str(band.source1_mask_cutout_path),
            'source2_mask_cutout_path': str(band.source2_mask_cutout_path),
            'psf_cutout_path': str(band.psf_cutout_path),
        }
        for band in bands
    },
    'parametric_main_mass': main_mass_summary(parametric_median, 'parametric'),
    'pixelated_main_mass': main_mass_summary(pixelated_median, 'pixelated'),
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / 'smoke_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(summary)
print('summary:', summary_path)
